# TP4 - Background Substraction
ATRIM - Option Datasim

Ecole Centrale Nantes

Diana Mateus

Participants: Denos Kume

#TODO
Figure out a **quantitative measurement** that makes a difference PSNR(w.r.t to a background empty region) CNR vs std curves such in the pet reconstruction, balance by the number of pixels , thicken the human masks

Try to see if they improve solution
- bilateral filtering
- skeletonisation
- Implement a brain mask to remove the border movement
- localized semi-automatic segmentation?- non-local means?
- change to use more or sliding window instead of fixed input image

Add to the metrics a weighted Dice score sklearn.metrics.f1_score

Implement as pipeline


### BACKGROUND SUBSTRACTION

The goal of this exercise is to enhance the video of a neurointervention  to improve the visualization of moving tools. To this end you will implement a pipeline of image processing methods to detect the moving tools automatically.


#### Methodology

As the brain is mostly static one way to detect the moving tools is to substract the background (first image) from each of the subsequent images. However, the results of this step need to be further improved. To this end, you will design a pipilene with the methods learnt in this course to produce a binary mask for the pixels that belong to the tools.

In your pipeline use at least:
- one histogram transformation
- one morphological operation
- one filtering operation in the spatial domain
- one filtering operation in the spectral domain
- one segmentation method

_The same pipeline should be applied to every image_


#### Expected output

The output of your image processing pipeline should be one binary image mask (with values 0 or 1) for every input image of the sequence, where
- the zero valued pixels indicate the moving tools inside each image.  
- the pixels with value 1 indicate the background (not a moving tool)

To validate the proposed method, a human has annotated (manually drawn) the tools of interest within the images. The annotated pixels belong either to catheters or guidewires. **Your masks should be as close as possible to the human annotations.**


#### Visualization of data and manual annotations

- Data visualization  (**do not include in final version**): visualize the neurointervention images in the ``` catheter``` folder with name ```frame_#```

- Individual Annotation visualization  (** do not include in final version**): visualize the manual annotations in the ```catheter``` folder with names ``` #_MicroCath``` and ```#_GuideWire```.

- Individual Annotation visualization  (** do not include in final version**): visualize the full manual annotation (union of the guidewire and microcatheter masks) by composing the union of the ``` #_MicroCath``` and ```#_GuideWire```. It should also be a binary mask.



#### Experimental (quantitative and qualitative  validation)

To compare your results and the manual annotations use the mean SAD (Sum of Absolute Differences) and the SNR (Signal to Noise Ratio) errors between your  mask and  the **full** manual mask.

Present the results qualitatively and quantitatively:

- Qualitatively:
     - Show your mask side by side with the manually annotated mask
     - Create an enhanced image suitable for guidance: enhance the contrast of the image and overlay your mask on the green channel of the enhanced image.

- Quantitatively:
    - compute and print the SAD (sum of absolute differences) error per image.
    - compute and print the MSE (sum of squared  differences ) error per image.
    - compute and print the PSNR (Peak signal to noise ratio) taking as reference image the manual annotations.
    - Then compute and print the mean and standard deviation of the three measures (SAD, MSE and PSNR) over the entire sequence.
    
Hints:
```
mse = numpy.mean( (img1 - img2) ** 2 )
PIXEL_MAX = 255.0 #or 1.0 or max over the signal of interest
psnr = 20 * math.log10(PIXEL_MAX / math.sqrt(mse))
```
or look at ```skimage.measure``` module

You may use modules such as ```scipy```, ```skimage``` or ``sklearn``(e.g. for clustering with K-means or a Gaussian Mixture Model). Ask me for other external modules.


## REPORT INSTRUCTIONS

#### 1. Intermediate Steps (Code and Description)
Report the results of the intermediate steps (when you add or remove a method from the pipeline):
- provide a text introduction with the idea that you intend to try
- show the implementation of the idea with code
- evaluate the quantitative and qualitative changes  when including, varying, adapting, etc the proposed method
- Discuss the scores or visualization improvements/degradations

#### 2. Final Pipeline (Code and Description)
Provide a detailed description of the best performing pipeline. Comment the code such that it is straightforward to relate the pipeline description to the code. Add your conclusions

- Describe the final retained pipeline
- Give a justification for every step (e.g. supported by experimental intermediate steps or theory).
- Add the **commented** code
- Display the qualitative and quantitative results
- Give your conclusions

REPORT:

In [1]:
import os
import numpy as np
#from scipy import misc
import skimage.io as io
from scipy import ndimage
from skimage.filters import gaussian
from skimage.filters import threshold_otsu
from skimage.morphology import erosion, dilation, opening, closing
from skimage.morphology import disk
from skimage.filters import threshold_otsu
from skimage import color
import fnmatch
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib as mpl

def linear (source, a, b):
    out = a*source+b
    return out


def saturate(source, perinf, persup):
    if perinf == 0:
        minper = np.min(source)
    else:
        minper=stats.scoreatpercentile(source,perinf)

    if persup == 100:
        maxper = np.max(source)
    else:
        maxper=stats.scoreatpercentile(source,persup)

    out = source
    out = (source-minper)/(maxper-minper)
    out[np.where(out>1)]=1.0
    out[np.where(out<0)]=0
    return out

def quadratic(source):
    out = (source)**2
    return out

def mse(im1,im2):
    return np.mean( (im1 - im2) ** 2 )

def sad(im1,im2):
    return np.mean(np.abs(im1 - im2))

def psnr(mse, pixel_max=255): #or 1.0
    return 20 * np.log10(pixel_max/ np.sqrt(mse))

def compute_errors(im1,im2):
    im1*=255
    im2*=255
    return sad(im1,im2), mse(im1,im2), psnr(mse(im1,im2))

In [2]:
# Local reproduction of the original notebook.
# Google Drive mounting is not required when the project is run from VS Code / WSL.
print("Local execution: Google Drive mount skipped.")

Local execution: Google Drive mount skipped.


In [ ]:
IMDIR = "../data/catheter"
print("IMDIR:", IMDIR)

IMDIR: ./data/catheter


In [6]:
error_dic = {'sad':[],'mse':[],'psnr':[]}

for num_f in range(201,292,10):

    # Build filenames from number sequence
    im_filename = ''.join(['frame_', str(num_f), '.png'])
    guidewire_filename = ''.join([str(num_f),'_GuideWire.tiff'])
    microcath_filename = ''.join([str(num_f),'_MicroCath.tiff'])

    im_f = os.path.join(IMDIR, im_filename)
    guidewire_f = os.path.join(IMDIR, guidewire_filename)
    microcath_f = os.path.join(IMDIR, microcath_filename)

    print(im_f)

    # Read image
    image = io.imread(im_f,as_gray=True)

    # Read guidewire
    guidewire_mask = io.imread(guidewire_f)
    if len(guidewire_mask.shape)>2:
        guidewire_mask = guidewire_mask[:,:,0]

    # Read microcatheter
    microcath_mask = io.imread(microcath_f)
    if len(microcath_mask.shape)>2:
        microcath_mask = microcath_mask[:,:,0]

    # Build a global mask from the guidewire and the microcatheter
    gt_mask = guidewire_mask
    gt_mask[np.where(microcath_mask<128)]=0

    #dilate mask to account for manual error
    selem = disk(2)
    gt_mask = erosion(gt_mask, selem)

    #make image and mask float to facilitate operations
    image = image.astype(float)/255.
    gt_mask = gt_mask.astype(float)/255.

    #filter and enhance input contrast
    im = image.copy()
    im = gaussian(im,sigma=1.0) #clearly improves SSD with sigma = 1.0
    #im = linear(im,3,0.1)
    im = saturate(im,0,90)

    #use first image as background model ->TODO: change to use more or sliding window
    if num_f == 201:
        im_ref = im.copy()
        continue
    else:

        #morphological filter on image difference to obtain mask
        im = -(im - im_ref)
        im_gauss_diff = im.copy() #Store to display later

        selem = disk(2)
        im = dilation(im, selem)

        #histogram transformation
        im = saturate(im,10,100)

        #thresholding
        im = im > 0.1
        #thresh = threshold_otsu(im) #Otsu does not work well for the first images, they seem to need further denoising
        #im = im > thresh

        #morphological
        selem = disk(2)
        im = opening(im, selem)
        im = 1-im
        im_mask = im.copy()

        #compute errors
        sad_i,mse_i,psnr_i = compute_errors(gt_mask,im_mask)
        error_dic['sad'].append(sad_i)
        error_dic['mse'].append(mse_i)
        error_dic['psnr'].append(psnr(mse_i))

        #Create 3 channel image for overlayed display
        im_show = color.gray2rgb(image)
        r_gt,c_gt = np.where(gt_mask<0.5)
        r_hat,c_hat = np.where(im_mask<0.5)
        im_show[r_gt,c_gt,1]=1 #gt
        im_show[r_hat,c_hat,0]=1 #estimate


    #Display results
    fig=plt.figure(figsize=(12,5))

    plt.subplot(1,5,1,adjustable='box')
    plt.imshow(image, cmap='gray')
    plt.title('original image')

    plt.subplot(1,5,2,adjustable='box')
    plt.imshow(im_gauss_diff,cmap='gray')
    plt.title('intermediate diff')

    plt.subplot(1,5,3)
    plt.imshow(im_mask, cmap='gray')
    plt.title('my mask')

    plt.subplot(1,5,4,adjustable='box')
    plt.imshow(gt_mask, cmap='gray')
    plt.title('gt_mask')

    plt.subplot(1,5,5,adjustable='box')
    plt.imshow(im_show)
    plt.title('overlay')

    plt.show()
    print('mean sad', error_dic['sad'][-1],
          'mean mse', error_dic['mse'][-1],
          'mean psnr', error_dic['psnr'][-1])



#total_error /=len(range(201,292,10))
#print('Mean Error:', np.mean(errors), 'Std: ', np.std(errors))



./data/catheter/frame_201.png


FileNotFoundError: [Errno 2] No such file or directory: '/home/denos/Master_SIP_EC-Nantes/Lab_Works/Image_Processing/Project/Background_Subtraction/notebooks/data/catheter/frame_201.png'

In [ ]:
print('Mean SAD', np.mean(error_dic['sad']),'+/-', np.std(error_dic['sad']))
print('Mean MSE', np.mean(error_dic['mse']),'+/-', np.std(error_dic['mse']))
print('Mean PSNR', np.mean(error_dic['psnr']),'+/-', np.std(error_dic['psnr']))
plt.subplot(1,3,1),  plt.plot(error_dic['sad']), plt.title('sad')
plt.subplot(1,3,2),  plt.plot(error_dic['mse']), plt.title('mse')
plt.subplot(1,3,3),  plt.plot(error_dic['psnr']), plt.title('psnr')
plt.show()

In [ ]:
#RESULTS WITH OTSU ->UNCOMMENT THE CODE IN PREVIOUS PIPELINE
print('Mean SAD', np.mean(error_dic['sad']),'+/-', np.std(error_dic['sad']))
print('Mean MSE', np.mean(error_dic['mse']),'+/-', np.std(error_dic['mse']))
print('Mean PSNR', np.mean(error_dic['psnr']),'+/-', np.std(error_dic['psnr']))
plt.subplot(1,3,1),  plt.plot(error_dic['sad']), plt.title('sad')
plt.subplot(1,3,2),  plt.plot(error_dic['mse']), plt.title('mse')
plt.subplot(1,3,3),  plt.plot(error_dic['psnr']), plt.title('psnr')
plt.show()

### 2. BACKGROUND SUBSTRACTION WITH EM

### 2.1 Expectation Maximisation

Run the following EM example where
1. Two 2D point clouds are randomly generated by sampling from  known Gaussian distributions (especified by their mean and covariance)
2. the EM algorithm is employed to fit a Gaussian Mixture Model (without knowing the parameters for generation)

Change the following parameters and write down the effect on the results
- the center and covariances of the Gaussian used during the generation
- compare the inferred parameters to those used in the generation as you vary the number of Gaussians, and later the number of samples
- print and describe what is the result of the gmm.predict(X) after applying gmm.fit(X) to a dataset X

#### Conclusions
Add your conclusions here

In [ ]:
import itertools
from scipy import linalg
from sklearn import mixture

color_iter = itertools.cycle(['navy', 'c', 'cornflowerblue', 'gold',
                              'darkorange'])


def plot_results(X, Y_, means, covariances, index, title):
    splot = plt.subplot(2, 1, 1 + index)
    for i, (mean, covar, color) in enumerate(zip(
            means, covariances, color_iter)):
        v, w = linalg.eigh(covar)
        v = 2. * np.sqrt(2.) * np.sqrt(v)
        u = w[0] / linalg.norm(w[0])

        # avoid plotting the redundant components.
        if not np.any(Y_ == i):
            continue
        plt.scatter(X[Y_ == i, 0], X[Y_ == i, 1], .8, color=color)

        # Plot an ellipse to show the Gaussian component
        angle = np.arctan(u[1] / u[0])
        angle = 180. * angle / np.pi  # convert to degrees
        ell = mpl.patches.Ellipse(mean, v[0], v[1], angle=180. + angle, color=color)
        ell.set_clip_box(splot.bbox)
        ell.set_alpha(0.5)
        splot.add_artist(ell)

    plt.xlim(-9., 5.)
    plt.ylim(-3., 6.)
    plt.title(title)


# Number of samples per component
n_samples = 500

# Generate random samples, two components
np.random.seed(0)
C = np.array([[0., -0.1], [1.7, .4]]) #covariance
X1 = np.dot(np.random.randn(n_samples, 2), C)
X2 = .7 * np.random.randn(n_samples, 2) + np.array([-6, 3])
X = np.vstack([X1,X2])

# Fit a Gaussian mixture with EM using five components
gmm = mixture.GaussianMixture(n_components=5,covariance_type='full')
gmm.fit(X)
plot_results(X, gmm.predict(X), gmm.means_, gmm.covariances_, 0,
             'Gaussian Mixture')

plt.show()


### 2.2 Background Substraction with Expectation Maximisation

Replace the thresholding in the Exercise 1 with an EM parameter estimation. The goal is to capture in one component the intensities belonging to the desired mask, and in the other the background. You can keep any elements of the pipeline that help improving the results.

Present the results as before qualitatively and quantitatively and compare them. Mention any other changes to the pipeline

``` Hint: ``` Scipy's gmm does not allow for  1D data change the shape using:

``` python
X = X.reshape((-1,1))
```



#### Changes to the pipeline and conclusions
(put here)

In [ ]:
error_dic = {'sad':[],'mse':[],'psnr':[]}

for num_f in range(201,292,10):

    # Build filenames from number sequence
    im_filename = ''.join(['frame_', str(num_f), '.png'])
    guidewire_filename = ''.join([str(num_f),'_GuideWire.tiff'])
    microcath_filename = ''.join([str(num_f),'_MicroCath.tiff'])

    im_f = os.path.join(IMDIR, im_filename)
    guidewire_f = os.path.join(IMDIR, guidewire_filename)
    microcath_f = os.path.join(IMDIR, microcath_filename)

    print(im_f)

    # Read image
    image = io.imread(im_f,as_gray=True)

    # Read guidewire
    guidewire_mask = io.imread(guidewire_f)
    if len(guidewire_mask.shape)>2:
        guidewire_mask = guidewire_mask[:,:,0]

    # Read microcatheter
    microcath_mask = io.imread(microcath_f)
    if len(microcath_mask.shape)>2:
        microcath_mask = microcath_mask[:,:,0]

    # Build a global mask from the guidewire and the microcatheter
    gt_mask = guidewire_mask
    gt_mask[np.where(microcath_mask<128)]=0

    #dilate mask to account for manual error
    selem = disk(2)
    gt_mask = erosion(gt_mask, selem)

    #make image and mask float to facilitate operations
    image = image.astype(float)/255.
    gt_mask = gt_mask.astype(float)/255.

    #filter and enhance input contrast
    im = image.copy()
    im = gaussian(im,sigma=1.0) #clearly improves SSD with sigma = 1.0
    #im = linear(im,3,0.1)
    im = saturate(im,0,90)


    #use first image as background model ->TODO: change to use more or sliding window
    if num_f == 201:
        im_ref = im.copy()
        continue
    else:

        #morphological filter on image difference to obtain mask
        im = -(im - im_ref)
        im_gauss_diff = im.copy() #Store to display later

        selem = disk(2)
        im = dilation(im, selem)

        #histogram transformation
        im = saturate(im,10,100)

        #thresholding
        #thresh = threshold_otsu(image)
        #im = im > thresh
        #im = im > 0.1

        #alternative EM mask
        gmm = mixture.GaussianMixture(n_components=2,covariance_type='full')
        X = im.reshape((-1,1))
        gmm.fit(X)
        assignments = gmm.predict(X)
        im_mask = np.reshape(assignments, np.shape(im))
        if (gmm.means_[0,0]<gmm.means_[1,0]):
            im_mask = 1-im_mask

        #show assignment histogram
        plt.hist(im_mask.ravel(), bins=100)
        plt.show()

        #morphological
        selem = disk(2)
        im_mask = opening(im_mask, selem)
        #im_mask = 1-im_mask
        #im_mask = im.copy()

        #compute errors
        sad_i,mse_i,psnr_i = compute_errors(gt_mask,im_mask)
        error_dic['sad'].append(sad_i)
        error_dic['mse'].append(mse_i)
        error_dic['psnr'].append(psnr(mse_i))

        #Create 3 channel image for overlayed display
        im_show = color.gray2rgb(image)
        r_gt,c_gt = np.where(gt_mask<0.5)
        r_hat,c_hat = np.where(im_mask<0.5)
        im_show[r_gt,c_gt,1]=1 #gt
        im_show[r_hat,c_hat,0]=1 #estimate


    #Display results
    fig=plt.figure(figsize=(12,5))

    plt.subplot(1,5,1,adjustable='box')
    plt.imshow(image, cmap='gray')
    plt.title('original image')

    plt.subplot(1,5,2,adjustable='box')
    plt.imshow(im_gauss_diff,cmap='gray')
    plt.title('intermediate diff')

    plt.subplot(1,5,3)
    plt.imshow(im_mask, cmap='gray')
    plt.title('my mask')

    plt.subplot(1,5,4,adjustable='box')
    plt.imshow(gt_mask, cmap='gray')
    plt.title('gt_mask')

    plt.subplot(1,5,5,adjustable='box')
    plt.imshow(im_show)
    plt.title('overlay')

    plt.show()
    print('mean sad', error_dic['sad'][-1],
          'mean mse', error_dic['mse'][-1],
          'mean psnr', error_dic['psnr'][-1])



#total_error /=len(range(201,292,10))
#print('Mean Error:', np.mean(errors), 'Std: ', np.std(errors))



In [ ]:
print('Mean SAD', np.mean(error_dic['sad']),'+/-', np.std(error_dic['sad']))
print('Mean MSE', np.mean(error_dic['mse']),'+/-', np.std(error_dic['mse']))
print('Mean PSNR', np.mean(error_dic['psnr']),'+/-', np.std(error_dic['psnr']))
plt.subplot(1,3,1),  plt.plot(error_dic['sad']), plt.title('sad')
plt.subplot(1,3,2),  plt.plot(error_dic['mse']), plt.title('mse')
plt.subplot(1,3,3),  plt.plot(error_dic['psnr']), plt.title('psnr')
plt.show()